# Multi-Judging LLM System for Exam Answer Evaluation
This notebook implements the architecture proposed for evaluating exam answers using Google's Gemini models.

In [ ]:
!pip install -q google-genai pandas scikit-learn seaborn matplotlib pydantic

## 1. Setup and Authentication
Provide your `GEMINI_API_KEY` in the Google Colab Secrets (the key icon on the left sidebar).

In [ ]:
import pandas as pd
import json
import statistics
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, cohen_kappa_score
from scipy.stats import pearsonr
from google import genai
from google.colab import userdata

# Initialize Gemini Client
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_NAME = 'gemini-2.0-flash-lite'

# Configuration
SELF_CONSISTENCY_RUNS = 3
WEIGHTS = {"moderate": 0.50, "strict": 0.25, "lenient": 0.25}

## 2. API Helper Function

In [ ]:
def call_gemini(prompt: str, retries: int = 5) -> str:
    """Call Gemini API with retry logic."""
    for attempt in range(retries):
        try:
            response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
            return response.text.strip()
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                time.sleep(30 * (attempt + 1))
            else:
                raise
    raise RuntimeError(f"API failed after {retries} retries.")

## 3. Preprocessing Module

In [ ]:
def preprocess(question, model_answer, student_answer):
    """Standardizes input formatting."""
    return {
        "question": str(question).strip(),
        "model_answer": str(model_answer).strip(),
        "student_answer": str(student_answer).strip()
    }

## 4. Rubric Generator Module
Extracts key concepts and precise marks from the reference answer.

In [ ]:
def generate_rubric(data: dict, total_marks: int) -> list:
    prompt = f"""You are an expert examiner. Extract the KEY CONCEPTS from the model answer and assign marks to each.
Question: {data['question']}
Model Answer: {data['model_answer']}
Total Marks: {total_marks}

Respond ONLY in this JSON format:
{{
  "rubric": [
    {{"concept": "...", "marks": int}},
    {{"concept": "...", "marks": int}}
  ]
}}"""
    raw = call_gemini(prompt)
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
    try:
        parsed = json.loads(raw.strip())
        return parsed.get("rubric", [])
    except json.JSONDecodeError:
        return [{"concept": "Overall answer quality", "marks": total_marks}]

def format_rubric(rubric_items: list) -> str:
    return "\n".join([f"  - {item['concept']} ({item['marks']} marks)" for item in rubric_items])

## 5. Multi-Judge Prompts & Evaluation
We use three distinct personas: Strict, Moderate, and Lenient.

In [ ]:
persona_instructions = {
    "strict": "You are a STRICT examiner. Penalize missing keywords and vague language. Deduct marks for any conceptual inaccuracy.",
    "moderate": "You are a MODERATE examiner. Allow reasonable paraphrasing. Award marks when the core idea is correct.",
    "lenient": "You are a LENIENT examiner. Focus only on whether the student grasped the core idea. Award marks generously."
}

def build_judge_prompt(persona: str, data: dict, rubric_items: list, total_marks: int) -> str:
    instruction = persona_instructions[persona]
    rubric_str = format_rubric(rubric_items)
    return f"""{instruction}
Evaluate the following student answer using the rubric below.
Question: {data['question']}
Model Answer: {data['model_answer']}
Rubric (Total: {total_marks} marks):
{rubric_str}
Student Answer: {data['student_answer']}

Respond ONLY in this JSON format:
{{
  "score": int,
  "covered_concepts": ["concept1"],
  "missing_concepts": ["concept2"],
  "feedback": "string",
  "reasoning": "string"
}}"""

def run_single_judge(persona: str, data: dict, rubric: list, total_marks: int) -> dict:
    prompt = build_judge_prompt(persona, data, rubric, total_marks)
    raw = call_gemini(prompt)
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
    try:
        result = json.loads(raw.strip())
        result['score'] = max(0, min(int(result.get('score', 0)), total_marks))
        result['persona'] = persona
        return result
    except Exception:
        return {"score": 0, "covered_concepts": [], "missing_concepts": [], "feedback": "Error", "reasoning": "", "persona": persona}

## 6. Self-Consistency Engine
Runs each judge multiple times (sampling via temperature > 0 theoretically, but Gemini allows multiple samples) to average the score.

In [ ]:
def run_judge_with_consistency(persona: str, data: dict, rubric: list, total_marks: int, runs: int = SELF_CONSISTENCY_RUNS) -> dict:
    scores, results, covered, missing, feedbacks = [], [], [], [], []
    for _ in range(runs):
        res = run_single_judge(persona, data, rubric, total_marks)
        scores.append(res['score'])
        results.append(res)
        covered.extend(res.get('covered_concepts', []))
        missing.extend(res.get('missing_concepts', []))
        time.sleep(1) # buffer
        
    avg_score = round(statistics.mean(scores), 2)
    std_score = round(statistics.stdev(scores) if len(scores) > 1 else 0.0, 2)
    best_run = min(results, key=lambda r: abs(r['score'] - avg_score))
    
    return {
        "persona": persona,
        "avg_score": avg_score,
        "std_dev": std_score,
        "covered_concepts": list(set(covered)),
        "missing_concepts": list(set(missing)),
        "feedback": best_run.get("feedback", ""),
        "reasoning": best_run.get("reasoning", "")
    }

## 7. Score Aggregation Module

In [ ]:
def aggregate_scores(judge_results: dict, total_marks: int) -> dict:
    s = judge_results["strict"]["avg_score"]
    m = judge_results["moderate"]["avg_score"]
    l = judge_results["lenient"]["avg_score"]
    
    final_score = round(WEIGHTS["moderate"] * m + WEIGHTS["strict"] * s + WEIGHTS["lenient"] * l, 2)
    final_score = max(0, min(final_score, total_marks))
    
    scores = [s, m, l]
    inter_var = round(statistics.variance(scores), 4) if len(scores) > 1 else 0.0
    disagreement = round(max(scores) - min(scores), 2)
    confidence = round((1 - disagreement / total_marks) * 100, 1) if total_marks > 0 else 100.0
    
    return {
        "final_score": round(final_score),
        "final_score_float": final_score,
        "strict_score": s,
        "moderate_score": m,
        "lenient_score": l,
        "inter_variance": inter_var,
        "disagreement": disagreement,
        "confidence_pct": confidence
    }

## 8. Feedback Synthesizer

In [ ]:
def generate_feedback(judge_results: dict, agg: dict, data: dict) -> dict:
    prompt = f"""Write a student-friendly feedback based on these judge scores.
Question: {data['question']}
Student Answer: {data['student_answer']}
Score: {agg['final_score_float']}
Covered: {judge_results['moderate']['covered_concepts']}
Missing: {judge_results['strict']['missing_concepts']}

Respond ONLY in this JSON format:
{{
  "strengths": "1-2 sentences",
  "missing_points": "1-2 sentences",
  "improvement": "1-2 sentences"
}}"""
    raw = call_gemini(prompt)
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except:
        return {"strengths": "Error", "missing_points": "", "improvement": ""}

## 9. Baseline Grader & Metrics

In [ ]:
def run_baseline(data: dict, total_marks: int) -> int:
    prompt = f"You are an examiner. Score this from 0 to {total_marks}. Reply ONLY with integer.\nQ:{data['question']}\nA:{data['model_answer']}\nStudent:{data['student_answer']}"
    raw = call_gemini(prompt)
    try:
        return max(0, min(int("".join(filter(str.isdigit, raw.split()[0]))), total_marks))
    except:
        return 0

def compute_metrics(human, multi, base):
    def safe_p(a, b): return round(pearsonr(a, b)[0], 4) if np.std(a)>0 and np.std(b)>0 else 0.0
    def safe_q(a, b): return round(cohen_kappa_score(np.round(a).astype(int), np.round(b).astype(int), weights="quadratic"), 4)
    return {
        "multi_judge": {"MAE": round(mean_absolute_error(human, multi), 4), "Pearson": safe_p(human, multi), "QWK": safe_q(human, multi)},
        "baseline": {"MAE": round(mean_absolute_error(human, base), 4), "Pearson": safe_p(human, base), "QWK": safe_q(human, base)}
    }

## 10. Execution Pipeline over Dataset
Please ensure `mec.csv` is uploaded to the Colab session.

In [ ]:
def process_dataset(df: pd.DataFrame, max_records: int = 5):
    records = df.head(max_records) if max_records else df
    results, human, multi, base = [], [], [], []
    
    for i, row in records.iterrows():
        print(f"\nProcessing [{i+1}/{len(records)}]...")
        total_m = int(row['total_marks'])
        data = preprocess(row['questions'], row['model_answer'], row['student_answer'])
        data['total_marks'] = total_m
        
        rubric = generate_rubric(data, total_m)
        jr = {
            "strict": run_judge_with_consistency("strict", data, rubric, total_m),
            "moderate": run_judge_with_consistency("moderate", data, rubric, total_m),
            "lenient": run_judge_with_consistency("lenient", data, rubric, total_m)
        }
        agg = aggregate_scores(jr, total_m)
        fb = generate_feedback(jr, agg, data)
        bs = run_baseline(data, total_m)
        
        teacher_score = int(row['teacher_marks']) if pd.notna(row['teacher_marks']) else None
        
        results.append({
            "question": data['question'],
            "student_answer": data['student_answer'],
            "teacher_score": teacher_score,
            "final_ai_score": agg['final_score'],
            "baseline_score": bs,
            "confidence": agg['confidence_pct'],
            "disagreement": agg['disagreement']
        })
        
        if teacher_score is not None:
            human.append(teacher_score)
            multi.append(agg['final_score'])
            base.append(bs)
            
        print(f"  -> Human: {teacher_score} | AI Final: {agg['final_score']} (Baseline: {bs}) | Confidence: {agg['confidence_pct']}%")
        
    df_res = pd.DataFrame(results)
    metrics = compute_metrics(human, multi, base) if len(human) >= 2 else None
    return df_res, metrics

# Load Dataset
print("Loading dataset...")
df = pd.read_csv("mec.csv")

# Run Pipeline on first 5 records
df_results, metrics = process_dataset(df, max_records=5)

print("\n=== Metrics ===")
print(metrics)